In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import joblib

In [2]:
df = pd.read_csv("../data/cognitive_impairment_dataset_2k.csv")

In [3]:
df.drop(columns=["Participant_ID"], inplace=True)

In [4]:
label_encoders = {}

categorical_cols = [
    "Gender", "Region", "Marital_Status",
    "Smoking_Status", "Alcohol_Use"
]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le


In [5]:
df["Cognitive_Impairment_Status"] = (
    (df["MMSE_Score"] < 24) | (df["GDS_Score"] > 10)
).astype(int)


In [6]:
X = df.drop(columns=["Cognitive_Impairment_Status"])
y = df["Cognitive_Impairment_Status"]
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)



RandomForestClassifier(class_weight='balanced', max_depth=8, min_samples_leaf=5,
                       min_samples_split=10, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [9]:
from sklearn.metrics import accuracy_score, classification_report

print("Training Accuracy:", accuracy_score(y_train, model.predict(X_train)))
print("Testing Accuracy :", accuracy_score(y_test, model.predict(X_test)))

print(classification_report(y_test, model.predict(X_test)))


Training Accuracy: 1.0
Testing Accuracy : 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       140
           1       1.00      1.00      1.00       301

    accuracy                           1.00       441
   macro avg       1.00      1.00      1.00       441
weighted avg       1.00      1.00      1.00       441



In [10]:
joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoders, "label_encoders.pkl")

['label_encoders.pkl']

In [11]:
print(df.head())

         Age  Gender  Education_Level  Region  Marital_Status  \
0  66.095267       0         1.007717       1               2   
1  79.056204       0         1.989415       1               3   
2  87.975645       0        -0.006862       1               1   
3  73.983815       0         1.988853       1               1   
4  70.223186       1         0.986624       1               3   

   Chronic_Diseases  Glucose_Level        BMI  MMSE_Score  GDS_Score  \
0         -0.011455     131.927553  22.230029   23.879322   1.190326   
1          3.990561     136.156526  22.923931   20.707593  11.598671   
2          4.002135      85.207140  31.450723   19.365337   2.874970   
3          1.992265     110.624706  28.462588   19.126083   7.834405   
4          2.002249     122.235395  29.102255   17.361092   7.763374   

   Sleep_Quality_Score  Physical_Activity_Score  Smoking_Status  Alcohol_Use  \
0             4.012028                 7.044219               0            1   
1             4.

In [12]:
print(df.tail())

            Age  Gender  Education_Level  Region  Marital_Status  \
2196  64.092133       1         1.998999       1               3   
2197  69.033103       0         3.003407       0               3   
2198  68.820785       0         0.988329       0               2   
2199  78.091628       0         1.007816       0               0   
2200  76.048607       0         1.007714       1               3   

      Chronic_Diseases  Glucose_Level        BMI  MMSE_Score  GDS_Score  \
2196          2.002478     138.475807  27.671815   26.542872  -0.002705   
2197          0.999115     132.203563  25.350667    3.239856   7.159314   
2198          2.970388     135.106215  23.983744   21.120823   3.947324   
2199          0.973232     139.646954  25.372889   24.959663   2.023450   
2200          2.993039     135.263952  29.132404   16.130188   2.471608   

      Sleep_Quality_Score  Physical_Activity_Score  Smoking_Status  \
2196             4.993119                 2.028207               0   


In [13]:
for col, encoder in label_encoders.items():
    print(col, encoder.classes_)

Gender ['Female' 'Male']
Region ['Rural' 'Urban']
Marital_Status ['Divorced' 'Married' 'Single' 'Widowed']
Smoking_Status ['No' 'Yes']
Alcohol_Use ['No' 'Yes']
